In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from collections import Counter
from itertools import combinations
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
warnings.filterwarnings("ignore")

In [29]:
# Load data
df = pd.read_csv("train.csv")

df = df[["itemId", "price"]].drop_duplicates().copy()

# normalize price
df["price"] = df["price"] / 100_000

In [6]:
item_price_count = (
    df.groupby("itemId")["price"]
      .nunique()
)

item_price_count.head(4)

itemId
66326595    1
70111495    2
70267256    1
70275797    1
Name: price, dtype: int64

In [7]:
single_item_ids = item_price_count[item_price_count == 1].index
multiple_item_ids = item_price_count[item_price_count > 1].index

df_item_single_price = df[df["itemId"].isin(single_item_ids)].copy()
df_item_multiple_price = df[df["itemId"].isin(multiple_item_ids)].copy()

In [8]:
df_item_single_price.head(4)

,itemId,price
10,841382440,390.0
18,22076783741,169.0
41,23286110847,35.0
42,15961274522,290.0


In [9]:
df_item_multiple_price.head(4)

,itemId,price
0,29302587134,419.0
1,18087284916,169.0
2,18087284916,119.0
7,18087284916,149.0


In [19]:
df_item_price_stats = (
    df_item_multiple_price
    .groupby("itemId")
    .agg(
        avg_price=("price", "mean"),
        max_price=("price", "max"),
        min_price=("price", "min"),
        std_price=("price", "std"),
        q1_price=("price", lambda x: x.quantile(0.25)),
        q3_price=("price", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)

df_item_price_stats["iqr_price"] = (
    df_item_price_stats["q3_price"] - df_item_price_stats["q1_price"]
)

df_item_price_stats["lower_bound"] = (
    df_item_price_stats["q1_price"] - 1.5 * df_item_price_stats["iqr_price"]
)

df_item_price_stats["upper_bound"] = (
    df_item_price_stats["q3_price"] + 1.5 * df_item_price_stats["iqr_price"]
)

In [20]:
len(df)

3162

In [21]:
df_item_price_stats

,itemId,avg_price,max_price,min_price,std_price,q1_price,q3_price,iqr_price,lower_bound,upper_bound
0,70111495,560.000000,565.0,555.0,7.071068,557.50,562.50,5.00,550.000,570.000
1,264594053,2110.000000,3300.0,1450.0,1032.618032,1515.00,2440.00,925.00,127.500,3827.500
2,265189303,802.333333,830.0,777.0,26.576932,788.50,815.00,26.50,748.750,854.750
3,268547497,231.250000,250.0,210.0,17.500000,221.25,242.50,21.25,189.375,274.375
4,276116347,550.333333,665.0,426.0,104.068567,464.00,636.25,172.25,205.625,894.625
...,...,...,...,...,...,...,...,...,...,...
633,29756524621,381.500000,530.0,290.0,112.113781,304.25,472.50,168.25,51.875,724.875
634,29759494159,619.500000,640.0,599.0,28.991378,609.25,629.75,20.50,578.500,660.500
635,29767848295,54.000000,59.0,49.0,7.071068,51.50,56.50,5.00,44.000,64.000
636,29808638945,229.000000,239.0,219.0,10.000000,224.00,234.00,10.00,209.000,249.000


In [22]:
df_new = pd.read_csv("train.csv")

itemIdList = (
    df_item_price_stats[df_item_price_stats["std_price"] > 3000]["itemId"]
    .tolist()
)

In [23]:
len(itemIdList)

2

In [24]:
df_check = df_new[df_new["itemId"].isin(itemIdList)]

In [25]:
len(df_check)

40

In [26]:
df_check.head(4)

,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,stock,normal_stock,...,item_price_max,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count,is_official_shop,is_verified,is_preferred_plus_seller
10848,2025-02-02 01:05:32.450,1005504584,22270840022,128673439540,599000000,0,0,100010,0.0,0.0,...,599000000,5.0,12,12,4.894366,53.0,264,f,f,f
10849,2025-02-02 01:05:32.450,1005504584,22270840022,128673439541,150000000,0,0,100010,0.0,0.0,...,599000000,5.0,12,12,4.894366,53.0,264,f,f,f
14077,2025-02-03 21:11:43.265,1005504584,22270840022,128673439540,599000000,0,0,100010,0.0,0.0,...,599000000,5.0,12,12,4.894366,55.0,263,f,f,f
14078,2025-02-03 21:11:43.265,1005504584,22270840022,128673439541,150000000,0,0,100010,0.0,0.0,...,599000000,5.0,12,12,4.894366,55.0,263,f,f,f


In [44]:
# Load data
df_new = pd.read_csv("train.csv")

# normalize price
df_new["price"] = df_new["price"] / 100_000

df_check = df_new.merge(
    df_item_price_stats[
        ["itemId", "lower_bound", "upper_bound"]
    ],
    on="itemId",
    how="left"
)

df_filter = df_check[
    (df_check["price"] < df_check["lower_bound"]) |
    (df_check["price"] > df_check["upper_bound"])
].copy()

In [46]:
df_filter

,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,stock,normal_stock,...,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count,is_official_shop,is_verified,is_preferred_plus_seller,lower_bound,upper_bound
182,2025-01-02 18:12:05.101,1005099271,24961844149,245300161784,480.0,0,0,100015,NaN,NaN,...,10,10,4.936721,67.0,139,f,f,f,83.000,203.000
339,2025-01-03 08:47:05.078,100689819,1643983672,151557885760,6490.0,0,0,100636,NaN,NaN,...,2,2,4.952811,84.0,1189,f,t,f,4258.750,6388.750
494,2025-01-03 10:37:08.374,100689819,1644045056,2738855557,2590.0,0,0,100636,NaN,NaN,...,7,7,4.952876,84.0,1189,f,t,f,2646.250,7836.250
574,2025-01-03 11:02:51.557,100314387,28255641382,166579186172,990.0,0,0,100632,NaN,NaN,...,12,12,4.985542,95.0,715,f,f,f,70.625,785.625
601,2025-01-03 11:36:12.077,100955244,24878384288,197483315911,588.0,0,0,100632,NaN,NaN,...,59,59,4.979947,98.0,221,f,f,f,103.000,543.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306119,2025-03-22 02:50:18.150,1003095770,19688868849,187919333180,1555.0,0,0,100630,NaN,NaN,...,40,40,4.950355,97.0,19,f,f,f,1901.000,2165.000
306120,2025-03-22 02:50:18.150,1003095770,19688868849,187919333180,1555.0,0,0,100630,NaN,NaN,...,40,40,4.950355,97.0,19,f,f,f,1901.000,2165.000
306122,2025-03-22 02:50:18.150,1003095770,19688868849,187919333181,1555.0,0,0,100630,NaN,NaN,...,40,40,4.950355,97.0,19,f,f,f,1901.000,2165.000
306189,2025-03-22 04:17:29.207,100964600,24052377201,240069696046,118.0,0,0,100629,NaN,NaN,...,47,47,4.986667,100.0,12,f,f,f,30.500,82.500
